In [1]:
# Cell 1: imports

import os
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from sklearn.preprocessing import StandardScaler

# Use your own indicator functions
from technicals.indicators import BollingerBands, ATR, RSI, MACD  # :contentReference[oaicite:2]{index=2}

print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.5.1+cu121
Lightning: 2.5.6
CUDA available: True


In [3]:
# Cell 2: load M5 data

DATA_FILE = Path(r"C:\Users\admin\Desktop\Forex_algo\code\data\EUR_USD_M5.parquet")

df = pd.read_parquet(DATA_FILE)
df = df.sort_values("time").reset_index(drop=True)


print(df.head())
print(df.dtypes)
print("Rows:", len(df))


                       time  volume    mid_o    mid_h    mid_l    mid_c  \
0 2016-01-07 00:00:00+00:00      74  1.07764  1.07811  1.07759  1.07786   
1 2016-01-07 00:05:00+00:00      98  1.07788  1.07818  1.07764  1.07810   
2 2016-01-07 00:10:00+00:00      28  1.07812  1.07832  1.07812  1.07828   
3 2016-01-07 00:15:00+00:00      40  1.07824  1.07830  1.07798  1.07798   
4 2016-01-07 00:20:00+00:00      53  1.07796  1.07799  1.07776  1.07790   

     bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
0  1.07757  1.07802  1.07750  1.07777  1.07772  1.07820  1.07768  1.07795  
1  1.07779  1.07811  1.07755  1.07802  1.07798  1.07827  1.07772  1.07819  
2  1.07803  1.07823  1.07803  1.07819  1.07822  1.07840  1.07822  1.07837  
3  1.07815  1.07821  1.07790  1.07790  1.07834  1.07838  1.07807  1.07807  
4  1.07787  1.07789  1.07768  1.07782  1.07804  1.07809  1.07784  1.07797  
time      datetime64[ns, UTC]
volume                  int64
mid_o                 float64
mid

In [4]:
# Cell 3: feature engineering + target

df_feat = df.copy()

# --- Indicators using your code ---
df_feat = RSI(df_feat, n=14)             # adds RSI_14
df_feat = MACD(df_feat)                  # adds MACD, SIGNAL, HIST
df_feat = ATR(df_feat, n=14)             # adds ATR_14
df_feat = BollingerBands(df_feat, n=20, s=2)  # adds BB_MA, BB_UP, BB_LW

# --- Time features (for intraday structure) ---
df_feat["time"] = pd.to_datetime(df_feat["time"])
df_feat["hour"] = df_feat["time"].dt.hour
df_feat["minute"] = df_feat["time"].dt.minute
df_feat["day_of_week"] = df_feat["time"].dt.dayofweek

# --- Basic price features / returns ---
df_feat["ret_1"] = df_feat["mid_c"].pct_change()
df_feat["ret_3"] = df_feat["mid_c"].pct_change(3)
df_feat["ret_6"] = df_feat["mid_c"].pct_change(6)

# --- Target: next-bar price change (in price units) ---
df_feat["target"] = df_feat["mid_c"].shift(-1) - df_feat["mid_c"]

# Drop rows with NaNs (from indicators / returns / target)
df_feat = df_feat.dropna().reset_index(drop=True)

print(df_feat[["time", "mid_c", "RSI_14", "MACD", "SIGNAL", "ATR_14", "BB_MA", "target"]].head())
print("Final rows after feature engineering:", len(df_feat))


                       time    mid_c     RSI_14      MACD    SIGNAL    ATR_14  \
0 2016-01-07 02:45:00+00:00  1.08146  69.310433  0.000572  0.000524  0.000635   
1 2016-01-07 02:50:00+00:00  1.08148  69.461247  0.000587  0.000538  0.000604   
2 2016-01-07 02:55:00+00:00  1.08152  69.781094  0.000595  0.000551  0.000510   
3 2016-01-07 03:00:00+00:00  1.08153  69.866065  0.000595  0.000560  0.000453   
4 2016-01-07 03:05:00+00:00  1.08176  71.828165  0.000606  0.000570  0.000420   

      BB_MA   target  
0  1.080179  0.00002  
1  1.080366  0.00004  
2  1.080527  0.00001  
3  1.080650  0.00023  
4  1.080756  0.00016  
Final rows after feature engineering: 735911


In [5]:
# Cell 4: pick feature columns & scale

FEATURE_COLS = [
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume",
    "RSI_14", "MACD", "SIGNAL", "HIST",
    "ATR_14", "BB_MA", "BB_UP", "BB_LW",
    "ret_1", "ret_3", "ret_6",
    "hour", "minute", "day_of_week"
]

TARGET_COL = "target"

X = df_feat[FEATURE_COLS].astype(np.float32).values
y = df_feat[TARGET_COL].astype(np.float32).values

n = len(df_feat)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:], y[val_end:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Train / Val / Test sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])


Train / Val / Test sizes: 515137 110387 110387


In [6]:
# Cell 5: PyTorch Dataset & DataLoader

SEQ_LEN = 96  # you can try 64, 96, 128

class ForexM5Dataset(Dataset):
    """
    Creates sequences of length SEQ_LEN and predicts the target at the last timestep.
    """
    def __init__(self, X, y, seq_len):
        self.X = X
        self.y = y
        self.seq_len = seq_len

        # valid indices: from seq_len-1 to len(X)-1 (last index has valid target)
        self.indices = np.arange(seq_len - 1, len(X))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        x_seq = self.X[t - self.seq_len + 1 : t + 1]    # [seq_len, features]
        y_t   = self.y[t]                               # scalar

        # convert to tensors
        x_seq = torch.from_numpy(x_seq)    # [seq_len, d_in]
        y_t   = torch.tensor(y_t, dtype=torch.float32)  # []

        return x_seq, y_t

train_ds = ForexM5Dataset(X_train_scaled, y_train, SEQ_LEN)
val_ds   = ForexM5Dataset(X_val_scaled,   y_val,   SEQ_LEN)
test_ds  = ForexM5Dataset(X_test_scaled,  y_test,  SEQ_LEN)

BATCH_SIZE = 256

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(train_ds), len(val_ds), len(test_ds)


(515042, 110292, 110292)

In [7]:
# Cell 6: Transformer model with Lightning

class ForexTransformer(pl.LightningModule):
    def __init__(
        self,
        d_in: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        lr: float = 1e-3,
    ):
        super().__init__()

        # loss fn defined first, then save hyperparams (ignore loss_fn object)
        self.loss_fn = nn.MSELoss()
        self.save_hyperparameters(ignore=["loss_fn"])

        self.input_proj = nn.Linear(d_in, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # [batch, seq, features]
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc_out = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: [batch, seq_len, d_in]
        x = self.input_proj(x)            # [batch, seq_len, d_model]
        enc_out = self.encoder(x)         # [batch, seq_len, d_model]
        last_token = enc_out[:, -1, :]    # [batch, d_model]
        out = self.fc_out(last_token)     # [batch, 1]
        return out.squeeze(-1)            # [batch]

    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.loss_fn(preds, y)

        mae = torch.mean(torch.abs(preds - y))
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_mae", mae, prog_bar=True, on_epoch=True)

        return {"val_loss": loss, "val_mae": mae}

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

d_in = len(FEATURE_COLS)
model = ForexTransformer(d_in=d_in, d_model=64, nhead=4, num_layers=3, dim_feedforward=128, dropout=0.1, lr=1e-3)
model


ForexTransformer(
  (loss_fn): MSELoss()
  (input_proj): Linear(in_features=19, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc_out): Linear(in_features=64, out_features=1, bias=True)
)

In [8]:
# Cell 7: training

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

trainer = pl.Trainer(
    max_epochs=20,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.1,
)

trainer.fit(model, train_loader, val_loader)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type               | Params | Mode 
----------------------------------------------------------
0 | loss_fn    | MSELoss            | 0      | train
1 | input_proj | Linear             | 1.3 K  | train
2 | encoder    | TransformerEncoder | 100 K  | train
3 | fc_out     | Linear   

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 19: 100%|██████████| 2012/2012 [00:58<00:00, 34.61it/s, v_num=28, train_loss_step=1.32e-7, val_loss=1.85e-7, val_mae=0.000357, train_loss_epoch=1.71e-7]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 2012/2012 [00:58<00:00, 34.59it/s, v_num=28, train_loss_step=1.32e-7, val_loss=1.85e-7, val_mae=0.000357, train_loss_epoch=1.71e-7]


In [9]:
# Cell 8: evaluation on test set (MAE / RMSE in pips)

model.eval()
all_y = []
all_pred = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(model.device)
        y_batch = y_batch.to(model.device)
        preds = model(x_batch)
        all_y.append(y_batch.cpu().numpy())
        all_pred.append(preds.cpu().numpy())

all_y = np.concatenate(all_y)
all_pred = np.concatenate(all_pred)

mae_price = np.mean(np.abs(all_pred - all_y))
rmse_price = np.sqrt(np.mean((all_pred - all_y)**2))

# EURUSD pips = price * 10_000
mae_pips = mae_price * 10_000
rmse_pips = rmse_price * 10_000

print(f"MAE (price): {mae_price:.8f}")
print(f"RMSE (price): {rmse_price:.8f}")
print(f"MAE (pips): {mae_pips:.3f}")
print(f"RMSE (pips): {rmse_pips:.3f}")


MAE (price): 0.00036827
RMSE (price): 0.00045666
MAE (pips): 3.683
RMSE (pips): 4.567


In [10]:
# Cell 9: save checkpoint for later real-time use

CKPT_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\eurusd_m5_transformer_v1.ckpt")

trainer.save_checkpoint(str(CKPT_PATH))
print("Saved Lightning checkpoint to:", CKPT_PATH)

# also save preprocessing config (scaler + features + seq_len)
PREPROC_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\eurusd_m5_transformer_preproc.npz")

np.savez(
    PREPROC_PATH,
    feature_cols=np.array(FEATURE_COLS),
    seq_len=np.array([SEQ_LEN]),
    scaler_mean=scaler.mean_,
    scaler_scale=scaler.scale_,
)

print("Saved preprocessing config to:", PREPROC_PATH)


Saved Lightning checkpoint to: C:\Users\admin\Desktop\Forex_algo\code\eurusd_m5_transformer_v1.ckpt
Saved preprocessing config to: C:\Users\admin\Desktop\Forex_algo\code\eurusd_m5_transformer_preproc.npz


In [11]:
import numpy as np

# all_y = actual target (price change)
# all_pred = model predictions

# Naive baseline: always predict no change (0)
naive_pred = np.zeros_like(all_y)

mae_model = np.mean(np.abs(all_pred - all_y))
mae_naive = np.mean(np.abs(naive_pred - all_y))

mae_model_pips = mae_model * 10_000
mae_naive_pips = mae_naive * 10_000

print(f"Model MAE:  {mae_model_pips:.3f} pips")
print(f"Naive MAE:  {mae_naive_pips:.3f} pips")
print(f"Improvement: {(1 - mae_model/mae_naive)*100:.1f}%")


Model MAE:  3.683 pips
Naive MAE:  2.000 pips
Improvement: -84.1%
